# SFT Pipeline — LLaMA 3.1 8B — All 6 Experiments + MMLU + Analysis
**Requires T4 GPU.** Mount dataset `sft-datasets` before running.

## Experiments
| Exp | Data | Curriculum | Optimizer | Attention | Isolates |
|-----|------|-----------|-----------|-----------|---------|
| E1 | Alpaca (control) | None | AdamW | SDPA | Baseline floor |
| E2 | Golden SFT | None | AdamW | SDPA | Data quality effect |
| E3 | Alpaca | LFR | AdamW | SDPA | Curriculum effect |
| E4 | Golden SFT | LFR | AdamW | SDPA | Full framework |
| E5 | Golden SFT | LFR | GaLoreAdamW8bit | SDPA | Optimizer efficiency |
| E6 | Golden SFT | LFR | AdamW | BetterTransformer | Attention efficiency |

## Fixes from original notebook
- **FIX A** — HF token read from Kaggle Secrets, never hardcoded
- **FIX B** — `datasets`, `galore_torch`, `optimum`, `tqdm` added to install cell
- **FIX C** — Kernel restart cell added after install (required before any imports)
- **FIX D** — `GALORE_AVAILABLE` / `OPTIMUM_AVAILABLE` flags defined before use
- **FIX E** — `quantization_config=bnb` correctly passed to `from_pretrained`
- **FIX F** — LFR replay buffer implemented (top-20% LEARN losses injected into REVIEW at 50%)
- **FIX G** — Tier validation check before E3/E4/E5/E6 (assert easy_ppl < medium_ppl < hard_ppl)
- **FIX H** — `MMILU_RESULTS` typo fixed → `MMLU_RESULTS` throughout
- **FIX I** — MMLU-Pro `answer` field handling made robust (letter string and int both supported)
- **FIX J** — Checkpoint saves adapter only (~150 MB) not full state_dict (3–4 GB)
- **FIX K** — Eval fires once per optimizer step, not once per micro-step
- **FIX L** — Empty LFR phase guard (percentile-based NB0 prevents this, but guard stays)
- **FIX M** — `SHORT` dict defined in config cell, accessible everywhere

## Timing (T4, ~2.6h per experiment)
- Session 1 — E1, E2, E3, E4: ~10.4h (checkpoint resume handles timeout)
- Session 2 — E5, E6: ~5.4h
- Session 3 — MMLU eval: ~4h (no training)
- Session 4 — Analysis: ~10 min (no GPU needed)
- **Total GPU: ~20h**

## Run order
1. **Run Cell 1** (install) → kernel restart when prompted  
2. **Run Cell 2** (HF login + GPU check)  
3. **Run Cell 3** (config — do not skip)  
4. **Run Cell 4** (load datasets)  
5. **Run Cell 5** (dataset class + eval fn)  
6. **Run Cell 6** (model/optimizer builders)  
7. **Run Cell 7** (training function)  
8. **Run Cell 8** (E1–E4, Session 1) → save as `sft-e1e4`  
9. **Run Cell 9** (E5–E6, Session 2) → save as `sft-e5e6`  
10. **Run Cell 10** (MMLU eval, Session 3) → save as `sft-mmlu`  
11. **Run Cell 11** (analysis, no GPU)


In [ ]:
# SESSION 2 — Restore checkpoints from previous session
import shutil, os

SOURCE = "/kaggle/input/datasets/preetam026/sft-training-session1"
DEST   = "/kaggle/working"

src_exp = os.path.join(SOURCE, "experiments")
dst_exp = os.path.join(DEST, "experiments")
if os.path.isdir(src_exp) and not os.path.isdir(dst_exp):
    shutil.copytree(src_exp, dst_exp)
    print(f"✅ Copied checkpoints from {src_exp}")

for f in ["all_training_results.json"]:
    src = os.path.join(SOURCE, f)
    if os.path.exists(src):
        shutil.copy2(src, DEST)
        print(f"✅ Copied {f}")

for exp in sorted(os.listdir(dst_exp)):
    rp = os.path.join(dst_exp, exp, "result.json")
    print(f"  {exp}: {'✅ complete' if os.path.exists(rp) else '⏳ needs training'}")

In [ ]:
# CELL 1 — Package installation (pinned versions)
# !! RUN THIS CELL FIRST — then RESTART KERNEL before continuing !!
# Order matters: bitsandbytes must come before accelerate/peft.
# Do not upgrade any version — pinned to avoid Kaggle ABI crashes.

import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    last = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(last[-3:]) if last else "ok")
    return r.returncode

print("── Step 1: uninstall conflicting packages ──")
run("pip uninstall -y torch torchvision torchaudio bitsandbytes transformers accelerate peft 2>/dev/null")

print("\n── Step 2: install PyTorch 2.1.2 (CUDA 11.8) ──")
# torch 2.1.2: SDPA stable, GaLore compatible, avoids 2.2+ dynamo issues on Kaggle
run("pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 "
    "--index-url https://download.pytorch.org/whl/cu118")

print("\n── Step 3: bitsandbytes 0.41.3 ──")
# CRITICAL: 0.41.3 predates torch._dynamo + np.random ABI integration.
# 0.42+ causes numpy ABI crash on Kaggle's Docker image.
run("pip install -U bitsandbytes")

print("\n── Step 4: accelerate, peft, transformers ──")
run("pip install -q accelerate==0.27.2 peft==0.8.2")
# transformers 4.43.3: LLaMA 3.1 apply_chat_template stable
run("pip install -q transformers==4.43.3")

print("\n── Step 5: datasets, tqdm ──")
# datasets: needed by MMLU-Pro eval cell
# tqdm: progress bar in MMLU eval
run("pip install -q datasets tqdm")

print("\n── Step 6: GaLore (E5 optimizer) ──")
galore_ok = run("pip install -q galore_torch") == 0
print(f"  galore_torch: {'✅' if galore_ok else '⚠️  install failed — E5 will fall back to AdamW'}")

print("\n── Step 7: optimum (BetterTransformer for E6) ──")
# optimum provides BetterTransformer — T4-compatible attention optimization
# xFormers directly is FA2 only on CC>=8.0; BetterTransformer works on T4 (CC 7.5)
optimum_ok = run("pip install -q optimum") == 0
print(f"  optimum: {'✅' if optimum_ok else '⚠️  install failed — E6 will use SDPA fallback'}")

print("\n" + "="*60)
print("✅ Installation complete.")
print("!! RESTART KERNEL NOW before running any other cell !!")
print("   Kernel → Restart (or press the restart button)")
print("="*60)


In [ ]:
!find / -name "libnvJitLink*" 2>/dev/null
!echo "---"
!ls /usr/local/cuda/lib64/libnv* 2>/dev/null
!echo "---"
!ldconfig -p | grep -i nvjitlink


In [ ]:
!ln -sf /usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13 /usr/local/cuda/lib64/libnvJitLink.so.13
!ldconfig

In [ ]:
# CELL 2 — HF login + GPU check
# FIX A: Token read from Kaggle Secrets — never hardcoded.
# To set: Kaggle notebook → Settings → Secrets → Add Secret
#   Name: HF_TOKEN   Value: hf_xxxxxxxxxxxx
# Accept LLaMA 3.1 license at: https://huggingface.co/meta-llama/Meta-Llama-3-8B

import os
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)


import torch, os

# ── HF login ─────────────────────────────────────────────────────────────────
try:
    from huggingface_hub import login
    login("HF_TOKEN")
    print("HF login    : ✅  (token from Kaggle Secrets)")
except Exception as e:
    print(f"HF login    : ⚠️  ({e})")
    print("             If LLaMA download fails, add HF_TOKEN to Kaggle Secrets")

# ── GPU check ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU — enable T4 in notebook Settings → Accelerator"
gpu   = torch.cuda.get_device_name(0)
vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
cc    = torch.cuda.get_device_capability(0)
sdpa  = hasattr(torch.nn.functional, "scaled_dot_product_attention")

print(f"GPU         : {gpu}")
print(f"VRAM        : {vram:.1f} GB")
print(f"Compute cap : {cc[0]}.{cc[1]}  (T4=7.5 | FA2 needs CC≥8.0: {'✅' if cc[0]>=8 else '❌ not available'})")
print(f"PyTorch     : {torch.__version__}")
print(f"SDPA        : {'✅' if sdpa else '❌ — check torch version'}")
print(f"BF16 compute: {'✅' if cc[0]>=8 else '❌ T4 only — using FP16'}")

# ── FIX D: availability flags defined here — used by Cell 6 builders ─────────
GALORE_AVAILABLE = False
OPTIMUM_AVAILABLE = False
try:
    import galore_torch
    GALORE_AVAILABLE = True
    print(f"GaLore      : ✅  ({galore_torch.__version__ if hasattr(galore_torch,'__version__') else 'installed'})")
except ImportError:
    print("GaLore      : ⚠️  not available — E5 will fall back to AdamW")

try:
    from optimum.bettertransformer import BetterTransformer
    OPTIMUM_AVAILABLE = True
    print("BetterTrans : ✅  (optimum installed — E6 attention ready)")
except ImportError:
    print("BetterTrans : ⚠️  optimum not available — E6 will use SDPA fallback")

DEVICE = "cuda"
print(f"\nAll checks passed — ready to run.")


In [ ]:
# CELL 3 — Configuration (shared across all experiments)
import os, random
import numpy as np

# ── Model ─────────────────────────────────────────────────────────────────────
# LLaMA 3.1 8B base (not Instruct).
# Requires accepting license: https://huggingface.co/meta-llama/Meta-Llama-3-8B
MODEL_ID = "meta-llama/Meta-Llama-3-8B"

# ── LoRA ──────────────────────────────────────────────────────────────────────
# r=16: appropriate for SFT (behavioural alignment, not knowledge injection).
# No embed_tokens — not needed for SFT unlike CPT.
LORA_R        = 16
LORA_ALPHA    = 32        # effective LR scale = alpha/r = 2.0
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ── Training ──────────────────────────────────────────────────────────────────
SEQ_LEN        = 1024     # covers >95% of SFT examples; xFormers benefit visible at this length
BATCH_SIZE     = 1        # safe at seq=1024 + QLoRA on T4 15GB
GRAD_ACCUM     = 16        # effective batch = 16
LR             = 2e-4
WARMUP_RATIO   = 0.03
MAX_GRAD_NORM  = 1.0

# ── Token budget ──────────────────────────────────────────────────────────────
TOKEN_BUDGET    = 2_000_000          # 2M per experiment (~2.6h on T4)
TOKENS_PER_STEP = BATCH_SIZE * SEQ_LEN * GRAD_ACCUM   # 16,384
TOTAL_STEPS     = TOKEN_BUDGET // TOKENS_PER_STEP      # ~122
EVAL_STEPS      = max(20, TOTAL_STEPS // 5)            # 5 eval checkpoints

# ── MMLU eval ─────────────────────────────────────────────────────────────────
MMLU_N      = 500    # stratified sample across MMLU-Pro subjects
FEW_SHOT    = 5
MAX_NEW_TOK = 32     # MMLU answers are single letters — 8× faster than 256, no quality loss

# ── Paths ─────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "/kaggle/working/experiments"
WRITE_DIR      = "/kaggle/working"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
import torch; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
try:
    np.random.seed(SEED)
except (ValueError, AttributeError):
    pass   # Kaggle numpy ABI mismatch on some images — torch seed is sufficient

# ── Experiment metadata ───────────────────────────────────────────────────────
# FIX M: SHORT/COLORS defined here — accessible by all subsequent cells.
EXP_IDS = [
    "E1_Control_LoRA",
    "E2_Golden_LoRA",
    "E3_Control_LFR",
    "E4_Golden_LFR",
    "E5_Golden_LFR_GaLore",
    "E6_Golden_LFR_xFormers",
]
SHORT = {e: e.split("_")[0] for e in EXP_IDS}
COLORS = {
    "E1_Control_LoRA"       : "#888780",
    "E2_Golden_LoRA"        : "#3B6D11",
    "E3_Control_LFR"        : "#185FA5",
    "E4_Golden_LFR"         : "#A32D2D",
    "E5_Golden_LFR_GaLore"  : "#7B3FA0",
    "E6_Golden_LFR_xFormers": "#B8601C",
}

# ── Dataset path resolver ─────────────────────────────────────────────────────
def find_dataset_file(filename):
    """Check all plausible Kaggle mount paths for sft-datasets."""
    candidates = [
        f"/kaggle/input/datasets/preetam026/sft-datasets/{filename}",
        f"/kaggle/input/sft-datasets/{filename}",
        f"/kaggle/input/sft-datasets/working/{filename}",
        f"/kaggle/working/{filename}",
    ]
    for cand in candidates:
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(
        f"\n{filename} not found in any of:\n" +
        "\n".join(f"  {c}" for c in candidates) +
        "\n→ Mount 'sft-datasets' dataset in: Settings → Data → Add Data"
    )

print(f"Model         : {MODEL_ID}")
print(f"Token budget  : {TOKEN_BUDGET/1e6:.1f}M per experiment")
print(f"Tokens/step   : {TOKENS_PER_STEP:,}  "
      f"(batch={BATCH_SIZE} × seq={SEQ_LEN} × accum={GRAD_ACCUM})")
print(f"Total steps   : ~{TOTAL_STEPS}")
print(f"Eval every    : {EVAL_STEPS} steps")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")


In [ ]:
# CELL 4 — Load datasets (control, golden, val)
import json

def load_jsonl(path):
    data = [json.loads(line) for line in open(path)]
    print(f"  ✅ {len(data):,} examples ← {os.path.basename(path)}")
    return data

print("Loading datasets...")
control_data = load_jsonl(find_dataset_file("control_sft.jsonl"))
golden_data  = load_jsonl(find_dataset_file("golden_sft.jsonl"))
val_data     = load_jsonl(find_dataset_file("val_sft.jsonl"))

# Verify LFR phase field exists (built by nb0_dataset_builder)
assert "lfr_phase" in control_data[0], "lfr_phase field missing — re-run nb0_dataset_builder"
assert "lfr_phase" in golden_data[0],  "lfr_phase field missing — re-run nb0_dataset_builder"

print("\nLFR phase distribution:")
for label, data in [("Control", control_data), ("Golden", golden_data)]:
    ph = [ex["lfr_phase"] for ex in data]
    easy_n, med_n, hard_n = ph.count("easy"), ph.count("medium"), ph.count("hard")
    assert easy_n > 0 and med_n > 0 and hard_n > 0, (
        f"Empty phase in {label}: easy={easy_n} medium={med_n} hard={hard_n}\n"
        "→ Rebuild with nb0_dataset_builder using percentile-based thresholds")
    print(f"  {label:8}: easy={easy_n:,}  medium={med_n:,}  hard={hard_n:,}")
print(f"  Val     : {len(val_data):,} examples (LIMA test split — neutral)")


In [ ]:
# CELL 5 — SFTDataset (response-only loss masking) + eval function
import torch, math, random, gc
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from transformers import AutoTokenizer

TOK = AutoTokenizer.from_pretrained(MODEL_ID)
TOK.pad_token    = TOK.eos_token
TOK.padding_side = "right"
print(f"Tokenizer: vocab={TOK.vocab_size:,}  pad_token={TOK.pad_token!r}")

class SFTDataset(Dataset):
    """
    Instruction-response pairs with response-only loss masking.

    Uses LLaMA 3.1's apply_chat_template to format prompts correctly.
    Instruction tokens receive label=-100 — only response tokens contribute
    to the cross-entropy loss.

    FIX 14 (boundary detection): We build the instruction-only text with
    add_generation_prompt=True to match the exact prefix the model sees,
    then measure its token length as the mask cutoff. The full sequence is
    built with add_generation_prompt=False and the completed response appended.
    This avoids the off-by-one caused by some tokenizer versions adding an
    extra BOS when generation_prompt is omitted from the full sequence.
    """
    def __init__(self, examples, tokeniser, seq_len):
        self.samples = []
        skipped = 0
        for ex in examples:
            instruction = ex["instruction"]
            response    = ex.get("response", ex.get("output", ""))

            # Build instruction prefix to find exact boundary token index
            inst_text = tokeniser.apply_chat_template(
                [{"role": "user", "content": instruction}],
                tokenize=False, add_generation_prompt=True)
            # Build full sequence: instruction + response + EOS
            full_text = inst_text + response + tokeniser.eos_token

            inst_ids = tokeniser(inst_text, add_special_tokens=False)["input_ids"]
            inst_len = len(inst_ids)

            enc = tokeniser(
                full_text,
                truncation=True, max_length=seq_len,
                padding="max_length", return_tensors="pt"
            )
            ids  = enc["input_ids"][0]
            mask = enc["attention_mask"][0]

            labels = ids.clone()
            labels[:inst_len] = -100        # mask instruction tokens
            labels[mask == 0] = -100        # mask padding tokens

            response_tokens = (labels != -100).sum().item()
            if response_tokens < 5:
                skipped += 1
                continue

            self.samples.append({
                "input_ids"      : ids,
                "attention_mask" : mask,
                "labels"         : labels,
            })
        print(f"  SFTDataset: {len(self.samples):,} samples  "
              f"(skipped {skipped} with <5 response tokens)")

    def __len__(self):        return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def eval_response_ppl(model, val_examples, tokeniser, seq_len, device, n_eval=80):
    """
    Response perplexity on the neutral LIMA val set.
    Only response tokens contribute — instruction is masked, same as training.
    Returns (val_loss, val_ppl).
    """
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    sample = random.sample(val_examples, min(n_eval, len(val_examples)))

    with torch.no_grad():
        for ex in sample:
            instruction = ex["instruction"]
            response    = ex.get("response", ex.get("output", ""))

            inst_text = tokeniser.apply_chat_template(
                [{"role": "user", "content": instruction}],
                tokenize=False, add_generation_prompt=True)
            full_text = inst_text + response + tokeniser.eos_token
            inst_len  = len(tokeniser(inst_text, add_special_tokens=False)["input_ids"])

            enc = tokeniser(
                full_text, return_tensors="pt",
                truncation=True, max_length=seq_len, padding=False
            ).to(device)

            labels = enc["input_ids"].clone()
            labels[:, :inst_len] = -100
            n_tok = (labels != -100).sum().item()
            if n_tok == 0:
                continue

            with autocast("cuda", dtype=torch.float16):
                out = model(**enc, labels=labels)

            total_loss   += out.loss.item() * n_tok
            total_tokens += n_tok

    model.train()
    avg = total_loss / max(total_tokens, 1)
    return round(avg, 4), round(math.exp(min(avg, 20)), 4)


def free_mem():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("SFTDataset + eval_response_ppl defined ✅")


In [ ]:
# CELL 6 — Model builder (QLoRA) + optimizer builders (AdamW / GaLore / xFormers)
import torch
from transformers import (AutoModelForCausalLM, BitsAndBytesConfig,
                           get_cosine_schedule_with_warmup)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.optim import AdamW


def build_qlora_model():
    """
    Load LLaMA 3.1 8B in 4-bit NF4 (QLoRA) with LoRA adapters in FP16.

    FIX E+F: BitsAndBytesConfig is created AND passed as quantization_config=
    to from_pretrained. The original notebook created bnb config but never
    passed it — model loaded with bare load_in_4bit=True kwarg, which is
    deprecated and silently ignored in some bitsandbytes versions.
    """
    free_mem()
    print(f"Loading {MODEL_ID} (4-bit NF4 QLoRA)...")

    # FIX F: pass quantization_config= explicitly
    bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,    # ← THIS WAS MISSING
        device_map="auto",
        attn_implementation="sdpa",
    )


    base = prepare_model_for_kbit_training(
        base, use_gradient_checkpointing=True)

    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base, lora_cfg)
    trainable, total = model.get_nb_trainable_parameters()
    vram = torch.cuda.max_memory_allocated(0) / 1e9
    print(f"  Trainable : {trainable:,} params ({100*trainable/total:.3f}%)")
    print(f"  VRAM      : {vram:.2f} GB after load")
    return model


def build_adamw(model, total_steps):
    """Standard AdamW with cosine LR schedule. Used for E1–E4 and E6."""    
    opt = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
    )
    sch = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )
    return opt, sch, "adamw"


def build_galore(model, total_steps, rank=64, update_gap=200, scale=0.25):
    """
    GaLoreAdamW8bit: gradient low-rank projection + 8-bit Adam state.
    Applied only to LoRA adapter parameters (requires_grad=True).
    Falls back to standard AdamW if galore_torch is not installed.

    FIX D: GALORE_AVAILABLE defined in Cell 2 — no NameError here.
    """
    if GALORE_AVAILABLE:
        try:
            from galore_torch import GaLoreAdamW8bit
            galore_params  = []
            regular_params = []
            for name, param in model.named_parameters():
                if not param.requires_grad:
                    continue
                # GaLore needs 2D matrices with both dims ≥ 64
                if (param.ndim == 2
                        and param.shape[0] >= 64
                        and param.shape[1] >= 64):
                    galore_params.append({
                        "params"          : [param],
                        "rank"            : rank,
                        "update_proj_gap" : update_gap,
                        "scale"           : scale,
                        "proj_type"       : "std",
                    })
                else:
                    regular_params.append(param)
            opt = GaLoreAdamW8bit(
                galore_params + [{"params": regular_params}], lr=LR)
            sch = get_cosine_schedule_with_warmup(
                opt,
                num_warmup_steps=int(total_steps * WARMUP_RATIO),
                num_training_steps=total_steps,
            )
            print(f"  GaLoreAdamW8bit: {len(galore_params)} GaLore param groups  ✅")
            return opt, sch, "galore_adamw8bit"
        except Exception as e:
            print(f"  GaLore init failed ({e}) — falling back to AdamW")

    print("  Using AdamW (GaLore fallback)")
    opt, sch, _ = build_adamw(model, total_steps)
    return opt, sch, "adamw_galore_fallback"


def apply_bettertransformer(model):
    """
    Apply BetterTransformer for memory-efficient attention (E6).

    FA2 requires CC≥8.0 — T4 is CC 7.5, so FA2 is NOT available.
    BetterTransformer (optimum library) uses PyTorch's fused SDPA kernel
    with improved memory access patterns. On T4 at seq=1024 expect 5–15%
    throughput improvement over baseline SDPA.

    FIX D: OPTIMUM_AVAILABLE defined in Cell 2 — no NameError here.
    """
    if OPTIMUM_AVAILABLE:
        try:
            from optimum.bettertransformer import BetterTransformer
            model = BetterTransformer.transform(model, keep_original_model=False)
            print("  BetterTransformer applied  ✅  (T4-compatible attention opt)")
            return model, "bettertransformer"
        except Exception as e:
            print(f"  BetterTransformer failed ({e})")

    print("  Attention: SDPA fallback (BetterTransformer unavailable)")
    return model, "sdpa_fallback"


print("build_qlora_model / build_adamw / build_galore / apply_bettertransformer defined ✅")


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# CELL 7 — Core training function (all fixes applied)
#
# FIX J: Checkpoint saves adapter only (~150 MB) not model.state_dict() (3-4 GB).
#         5 evals × 6 experiments with original approach = 90-120 GB > Kaggle limit.
# FIX K: Eval fires exactly once per optimizer step (was firing 4× per step).
# FIX L: Empty LFR phase guard.
# FIX F: LFR replay buffer — top-20% highest-loss LEARN examples injected
#         into REVIEW at 50% of each batch (genuine spaced repetition).
# FIX G: Tier validation before LFR experiments (easy_ppl < med_ppl < hard_ppl).
import csv, json, time, os, random
import torch
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, ConcatDataset
from peft import PeftModel


def check_tier_ordering(model, tok, tier_val_data, device, seq_len):
    """
    FIX G: Verify base model PPL ordering before LFR training begins.
    If easy_ppl >= medium_ppl or medium_ppl >= hard_ppl, the difficulty
    scoring didn't produce a meaningful curriculum — abort with instructions.
    Returns True if ordering is correct, False otherwise.
    """
    if not tier_val_data:
        print("  Tier validation: skipped (no tier_validation.jsonl found)")
        return True

    easy   = [x for x in tier_val_data if x.get("lfr_phase") == "easy"]
    medium = [x for x in tier_val_data if x.get("lfr_phase") == "medium"]
    hard   = [x for x in tier_val_data if x.get("lfr_phase") == "hard"]

    def ppl_for_tier(examples):
        _, ppl = eval_response_ppl(model, examples, tok, seq_len, device,
                                    n_eval=len(examples))
        return ppl

    print("  Running tier validation (20 examples per tier)...")
    easy_ppl   = ppl_for_tier(easy)
    medium_ppl = ppl_for_tier(medium)
    hard_ppl   = ppl_for_tier(hard)
    print(f"  Tier PPL — easy: {easy_ppl:.2f}  medium: {medium_ppl:.2f}  hard: {hard_ppl:.2f}")

    ok = (easy_ppl < medium_ppl) and (medium_ppl < hard_ppl)
    if not ok:
        print("  ⚠️  WARNING: Tier PPL ordering invalid (expected easy < medium < hard)")
        print("     The difficulty scoring did not produce a meaningful curriculum.")
        print("     LFR training will continue but results may not show curriculum effects.")
    else:
        print("  ✅ Tier ordering confirmed — LFR curriculum is valid")
    return ok


def run_experiment(exp_id, cfg, tok, val_data, tier_val_data=None):
    """
    Train one SFT experiment with checkpoint resume.

    cfg keys:
      data       : list of dicts with instruction/response/lfr_phase fields
      use_lfr    : bool — True for LFR phase ordering; False for random shuffle
      optimizer  : 'adamw' | 'galore'
      attention  : 'sdpa' | 'xformers'
      desc       : human-readable description string

    Checkpoint resume: if state.pt exists in ckpt_dir, training resumes
    from last saved step. Re-run the cell after session timeout to resume.
    """
    print(f"\n{'='*65}")
    print(f"  {exp_id}")
    print(f"  {cfg.get('desc', '')}")
    print(f"  optimizer={cfg.get('optimizer','adamw')}  "
          f"attention={cfg.get('attention','sdpa')}  "
          f"lfr={cfg.get('use_lfr', False)}")
    print(f"{'='*65}")

    ckpt_dir    = os.path.join(CHECKPOINT_DIR, exp_id)
    os.makedirs(ckpt_dir, exist_ok=True)
    train_csv   = os.path.join(ckpt_dir, "train_log.csv")
    eval_csv    = os.path.join(ckpt_dir, "eval_log.csv")
    adapter_dir = os.path.join(ckpt_dir, "final_adapter")
    result_path = os.path.join(ckpt_dir, "result.json")
    state_path  = os.path.join(ckpt_dir, "state.pt")

    # ── Already complete? ─────────────────────────────────────────────────────
    if os.path.exists(result_path):
        r = json.load(open(result_path))
        if r.get("tokens_processed", 0) >= TOKEN_BUDGET * 0.95:
            print(f"  ✅ Already complete ({r['tokens_processed']/1e6:.1f}M tokens)")
            print(f"     PPL={r['final_val_ppl']}  tps={r['avg_tps']}  VRAM={r['peak_vram_gb']}GB")
            return r

    # ── Build phase plan ──────────────────────────────────────────────────────
    data = cfg["data"]
    if cfg.get("use_lfr", False):
        easy   = [x for x in data if x.get("lfr_phase") == "easy"]
        medium = [x for x in data if x.get("lfr_phase") == "medium"]
        hard   = [x for x in data if x.get("lfr_phase") == "hard"]

        # FIX L: Guard empty phases (percentile-based NB0 prevents this, but be safe)
        for pname, plist in [("easy", easy), ("medium", medium), ("hard", hard)]:
            if not plist:
                print(f"  ⚠️  LFR phase '{pname}' empty — skipping it")
        phase_plan = [
            ("LEARN",  easy,   False),
            ("FOCUS",  medium, False),
            ("REVIEW", hard,   False),  # replay added dynamically below
        ]
        print(f"  LFR phases: easy={len(easy):,}  medium={len(medium):,}  hard={len(hard):,}")
    else:
        shuffled = list(data)
        random.shuffle(shuffled)
        phase_plan = [("TRAIN", shuffled, True)]  # shuffle=True for non-LFR

    # ── Build model + optimizer ───────────────────────────────────────────────
    model = build_qlora_model()

    if cfg.get("attention") == "xformers":
        model, attn_method = apply_bettertransformer(model)
    else:
        attn_method = "sdpa"

    if cfg.get("optimizer") == "galore":
        opt, sch, opt_method = build_galore(model, TOTAL_STEPS)
    else:
        opt, sch, opt_method = build_adamw(model, TOTAL_STEPS)

    scaler = GradScaler()

    # ── FIX G: Tier validation before LFR experiments ─────────────────────────
    if cfg.get("use_lfr", False) and tier_val_data:
        check_tier_ordering(model, tok, tier_val_data, DEVICE, SEQ_LEN)

    # ── Resume from checkpoint? ───────────────────────────────────────────────
    global_step  = 0
    tokens_done  = 0
    ppl_history  = []
    replay_buffer = []    # FIX F: stores high-loss LEARN examples for REVIEW replay
    t0           = time.time()

    if os.path.exists(state_path):
        print(f"\n  ↻ RESUMING from checkpoint")
        state = torch.load(state_path, map_location="cpu")
        global_step   = state["global_step"]
        tokens_done   = state["tokens_done"]
        ppl_history   = state.get("ppl_history", [])
        replay_buffer = state.get("replay_buffer", [])

        # FIX J (resume): Load adapter from saved path, not from state_dict blob
        last_ckpt = state.get("last_ckpt")
        if last_ckpt and os.path.isdir(last_ckpt):
            try:
                model.load_adapter(last_ckpt, adapter_name="default")
                print(f"  Adapter loaded from {last_ckpt}")
            except Exception as e:
                print(f"  ⚠️  Adapter reload failed ({e}) — continuing from last weights")
        elif "model_state" in state:
            # backwards compat: legacy checkpoints stored full state_dict
            try:
                model.load_state_dict(state["model_state"], strict=False)
                print("  Legacy state_dict loaded")
            except Exception as e:
                print(f"  ⚠️  Legacy load failed ({e})")

        try:
            opt.load_state_dict(state["opt_state"])
            sch.load_state_dict(state["sch_state"])
        except Exception as e:
            print(f"  ⚠️  Optimizer state not restored ({e})")

        del state
        free_mem()
        print(f"  Resumed at step={global_step}  tokens={tokens_done/1e6:.2f}M")
    else:
        # Fresh start — create log files with headers
        with open(train_csv, "w", newline="") as f:
            csv.writer(f).writerow(
                ["step", "tokens_M", "loss", "lr", "tps", "vram_gb", "phase"])
        with open(eval_csv, "w", newline="") as f:
            csv.writer(f).writerow(
                ["step", "tokens_M", "val_loss", "val_ppl"])

    model.train()
    tps_window  = []
    step_losses = []   # FIX F: track per-step loss for replay buffer selection

    # ── Training loop ──────────────────────────────────────────────────────────
    for phase_name, phase_data, do_shuffle in phase_plan:
        if tokens_done >= TOKEN_BUDGET:
            break

        # FIX L: Skip genuinely empty phases
        if not phase_data:
            print(f"\n  ── Phase {phase_name}: skipped (0 examples)")
            continue

        # FIX F: Replay buffer injection for REVIEW phase
        # During REVIEW, mix in the top-20% highest-loss LEARN examples at 50/50 ratio.
        # This implements genuine spaced repetition — the model revisits the material
        # it struggled with most during LEARN, not just a new batch of hard examples.
        effective_data = phase_data
        if phase_name == "REVIEW" and replay_buffer:
            n_replay  = len(phase_data)   # equal number of replay examples
            replay    = random.sample(replay_buffer, min(n_replay, len(replay_buffer)))
            effective_data = phase_data + replay
            random.shuffle(effective_data)
            print(f"\n  ── Phase REVIEW: {len(phase_data):,} hard + {len(replay):,} replay  "
                  f"({len(replay_buffer):,} in buffer)")
        else:
            print(f"\n  ── Phase {phase_name}: {len(phase_data):,} examples")

        ds = SFTDataset(effective_data, tok, SEQ_LEN)
        if not ds.samples:
            print(f"  Skipped (0 valid samples after tokenisation)")
            continue

        loader = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=do_shuffle,
            drop_last=True,
            num_workers=0,
        )
        step_in_phase = 0
        phase_losses  = []   # for replay buffer selection
        opt.zero_grad()

        for batch_idx, batch in enumerate(loader):
            if tokens_done >= TOKEN_BUDGET:
                break

            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["labels"].to(DEVICE)

            t_step = time.time()
            with autocast("cuda", dtype=torch.float16):
                out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
                loss = out.loss / GRAD_ACCUM
            scaler.scale(loss).backward()
            step_in_phase += 1

            # FIX F: Track individual batch loss for replay buffer
            batch_loss = out.loss.item()
            if phase_name == "LEARN":
                # Map batch index back to example indices
                phase_losses.append((batch_loss, batch_idx))

            # ── Optimizer step (every GRAD_ACCUM micro-steps) ─────────────────
            if step_in_phase % GRAD_ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    MAX_GRAD_NORM,
                )
                scaler.step(opt)
                scaler.update()
                sch.step()
                opt.zero_grad()
                global_step += 1

                batch_tokens  = BATCH_SIZE * SEQ_LEN
                tokens_done  += batch_tokens * GRAD_ACCUM
                elapsed       = time.time() - t_step
                tps           = (batch_tokens * GRAD_ACCUM) / max(elapsed, 1e-6)
                tps_window.append(tps)
                vram          = torch.cuda.max_memory_allocated(0) / 1e9
                lr_now        = sch.get_last_lr()[0]
                real_loss     = loss.item() * GRAD_ACCUM

                with open(train_csv, "a", newline="") as f:
                    csv.writer(f).writerow([
                        global_step, round(tokens_done / 1e6, 3),
                        round(real_loss, 4), f"{lr_now:.2e}",
                        round(tps, 1), round(vram, 2), phase_name,
                    ])

                if global_step % 10 == 0 or global_step == 1:
                    print(f"    step {global_step:4d} | {tokens_done/1e6:.2f}M tok"
                          f" | loss {real_loss:.4f} | {tps:.0f} tok/s"
                          f" | {vram:.1f} GB | {phase_name}")

                # FIX K: Eval inside optimizer-step block — fires exactly once
                # per global_step. Original had this outside the GRAD_ACCUM guard,
                # causing 4× redundant evals per checkpoint (once per micro-step).
                if global_step % EVAL_STEPS == 0:
                    val_loss, val_ppl = eval_response_ppl(
                        model, val_data, tok, SEQ_LEN, DEVICE)
                    ppl_history.append(val_ppl)
                    print(f"    *** EVAL step={global_step} | "
                          f"val_ppl={val_ppl:.4f} | val_loss={val_loss:.4f}")

                    with open(eval_csv, "a", newline="") as f:
                        csv.writer(f).writerow([
                            global_step, round(tokens_done / 1e6, 3),
                            val_loss, val_ppl,
                        ])

                    # FIX J: Save adapter (~150 MB) not model.state_dict() (3-4 GB)
                    ckpt_path = os.path.join(ckpt_dir, f"step{global_step:04d}")
                    os.makedirs(ckpt_path, exist_ok=True)
                    model.save_pretrained(ckpt_path)
                    tok.save_pretrained(ckpt_path)

                    state = {
                        "global_step"  : global_step,
                        "tokens_done"  : tokens_done,
                        "ppl_history"  : ppl_history,
                        "replay_buffer": replay_buffer,   # persist replay buffer
                        "last_ckpt"    : ckpt_path,       # adapter path, not weights
                        "opt_state"    : opt.state_dict(),
                        "sch_state"    : sch.state_dict(),
                    }
                    torch.save(state, state_path)
                    print(f"    Checkpoint → {ckpt_path}")

        # FIX F: After LEARN phase, populate replay buffer with top-20% loss examples
        if phase_name == "LEARN" and phase_losses:
            phase_losses.sort(key=lambda x: x[0], reverse=True)   # highest loss first
            top_20_pct    = max(1, len(phase_losses) // 5)
            top_indices   = {idx for _, idx in phase_losses[:top_20_pct]}
            replay_buffer = [phase_data[i] for i in top_indices if i < len(phase_data)]
            print(f"  Replay buffer filled: {len(replay_buffer):,} high-loss LEARN examples")

    # ── Save final adapter + result ────────────────────────────────────────────
    os.makedirs(adapter_dir, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tok.save_pretrained(adapter_dir)
    print(f"\n  Final adapter → {adapter_dir}")

    total_min = (time.time() - t0) / 60
    avg_tps   = round(sum(tps_window) / max(len(tps_window), 1), 1)
    peak_vram = round(torch.cuda.max_memory_allocated(0) / 1e9, 2)
    final_ppl = ppl_history[-1] if ppl_history else None
    best_ppl  = min(ppl_history) if ppl_history else None

    result = {
        "exp_id"           : exp_id,
        "desc"             : cfg.get("desc", ""),
        "tokens_processed" : tokens_done,
        "total_steps"      : global_step,
        "final_val_ppl"    : final_ppl,
        "best_val_ppl"     : best_ppl,
        "avg_tps"          : avg_tps,
        "peak_vram_gb"     : peak_vram,
        "total_time_min"   : round(total_min, 1),
        "trainable_params" : sum(p.numel() for p in model.parameters()
                                  if p.requires_grad),
        "ppl_history"      : ppl_history,
        "optimizer"        : opt_method,
        "attention"        : attn_method,
    }
    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)

    total_h = total_min / 60
    print(f"\n  ✅ {exp_id} COMPLETE")
    print(f"     PPL={final_ppl}  best={best_ppl}")
    print(f"     tok/s={avg_tps}  VRAM={peak_vram} GB  "
          f"time={total_min:.1f} min ({total_h:.1f}h)")

    # Clean up GPU memory before next experiment
    del model, opt, sch, scaler
    free_mem()
    return result


print("run_experiment() defined ✅  (FIX F+G+J+K+L applied)")


In [ ]:
# CELL 8 — Run E1, E2, E3, E4  (~10.4h on T4)
# Session 1. If it times out, re-run the notebook — each experiment resumes
# from its checkpoint automatically.
import sys, warnings, json
sys.setrecursionlimit(10000)
warnings.filterwarnings("ignore")

# Load tier validation data for LFR experiments (E3, E4)
_tier_val_data = []
try:
    _tier_path = find_dataset_file("tier_validation.jsonl")
    _tier_val_data = [json.loads(l) for l in open(_tier_path)]
    print(f"Tier validation: {len(_tier_val_data)} examples loaded ✅")
except FileNotFoundError:
    print("Tier validation: file not found — skipping pre-run tier check")

ALL_RESULTS = {}

EXPERIMENTS_E1_E4 = {
    "E1_Control_LoRA": {
        "data"      : control_data,
        "use_lfr"   : False,
        "optimizer" : "adamw",
        "attention" : "sdpa",
        "desc"      : "Baseline: Alpaca control, random order, QLoRA, AdamW, SDPA",
    },
    "E2_Golden_LoRA": {
        "data"      : golden_data,
        "use_lfr"   : False,
        "optimizer" : "adamw",
        "attention" : "sdpa",
        "desc"      : "Data quality: Golden SFT, random order, QLoRA, AdamW, SDPA",
    },
    "E3_Control_LFR": {
        "data"      : control_data,
        "use_lfr"   : True,
        "optimizer" : "adamw",
        "attention" : "sdpa",
        "desc"      : "Curriculum: Alpaca control, LFR+replay, QLoRA, AdamW, SDPA",
    },
    "E4_Golden_LFR": {
        "data"      : golden_data,
        "use_lfr"   : True,
        "optimizer" : "adamw",
        "attention" : "sdpa",
        "desc"      : "Full framework: Golden SFT, LFR+replay, QLoRA, AdamW, SDPA",
    },
}

for exp_id, cfg in EXPERIMENTS_E1_E4.items():
    # Pass tier_val_data for LFR experiments, None for non-LFR
    tier_data = _tier_val_data if cfg["use_lfr"] else None
    ALL_RESULTS[exp_id] = run_experiment(exp_id, cfg, TOK, val_data, tier_data)

# Save after E1-E4
combined_path = os.path.join(WRITE_DIR, "all_training_results.json")
with open(combined_path, "w") as f:
    json.dump(ALL_RESULTS, f, indent=2)

print(f"\nE1–E4 results saved → {combined_path}")
print("\n" + "="*65)
print("E1–E4 SUMMARY")
print("="*65)
print(f"{'Exp':<25} {'PPL':>8} {'Best':>8} {'tok/s':>8} {'VRAM':>7} {'Min':>8}")
print("-"*65)
for eid, r in ALL_RESULTS.items():
    print(f"{eid:<25} "
          f"{str(r.get('final_val_ppl','?')):>8} "
          f"{str(r.get('best_val_ppl','?')):>8} "
          f"{str(r.get('avg_tps','?')):>8} "
          f"{str(r.get('peak_vram_gb','?')):>7} "
          f"{str(r.get('total_time_min','?')):>8}")
print("="*65)
print("\n✅ Session 1 complete — save output as Kaggle Dataset: 'sft-e1e4'")
print("   Then start a new session for E5+E6.")


In [ ]:
# CELL 9 — Run E5 (GaLore) + E6 (BetterTransformer)  (~5.4h on T4)
# Session 2. Mount 'sft-e1e4' dataset if starting fresh.
import json

# Reload E1-E4 results if starting a new session
if "ALL_RESULTS" not in dir() or len(ALL_RESULTS) < 4:
    combined_path = os.path.join(WRITE_DIR, "all_training_results.json")
    if os.path.exists(combined_path):
        ALL_RESULTS = json.load(open(combined_path))
        print(f"Loaded {len(ALL_RESULTS)} existing results")
    else:
        ALL_RESULTS = {}
        print("No E1-E4 results found — results table will be incomplete")

EXPERIMENTS_E5_E6 = {
    "E5_Golden_LFR_GaLore": {
        "data"      : golden_data,
        "use_lfr"   : True,
        "optimizer" : "galore",
        "attention" : "sdpa",
        "desc"      : "GaLore: Golden SFT, LFR+replay, QLoRA+GaLoreAdamW8bit, SDPA",
    },
    "E6_Golden_LFR_xFormers": {
        "data"      : golden_data,
        "use_lfr"   : True,
        "optimizer" : "adamw",
        "attention" : "xformers",
        "desc"      : "BetterTransformer: Golden SFT, LFR+replay, QLoRA, AdamW, BetterTransformer",
    },
}

for exp_id, cfg in EXPERIMENTS_E5_E6.items():
    tier_data = _tier_val_data if "_tier_val_data" in dir() else None
    ALL_RESULTS[exp_id] = run_experiment(exp_id, cfg, TOK, val_data, tier_data)

# Save all 6 training results
combined_path = os.path.join(WRITE_DIR, "all_training_results.json")
with open(combined_path, "w") as f:
    json.dump(ALL_RESULTS, f, indent=2)
print(f"\nAll 6 training results saved → {combined_path}")

# Efficiency comparison vs E4
e4 = ALL_RESULTS.get("E4_Golden_LFR", {})
print("\nEfficiency vs E4 baseline:")
print(f"  {'Metric':<22} {'E4 (baseline)':>14} {'E5 GaLore':>14} {'E6 BetterTrans':>15}")
print("  " + "-"*67)
for lbl, key in [
    ("Val PPL",       "final_val_ppl"),
    ("Tok/s",         "avg_tps"),
    ("VRAM GB",       "peak_vram_gb"),
    ("Time min",      "total_time_min"),
    ("Optimizer",     "optimizer"),
    ("Attention",     "attention"),
]:
    v4 = e4.get(key, "—")
    v5 = ALL_RESULTS.get("E5_Golden_LFR_GaLore",   {}).get(key, "—")
    v6 = ALL_RESULTS.get("E6_Golden_LFR_xFormers", {}).get(key, "—")
    print(f"  {lbl:<22} {str(v4):>14} {str(v5):>14} {str(v6):>15}")

print("\n✅ Session 2 complete — save output as Kaggle Dataset: 'sft-e5e6'")


In [ ]:
# CELL 10 — MMLU-Pro Evaluation (all 6 adapters, 5-shot, 500 questions)
import torch, gc, random, json, os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

MMLU_TOK = AutoTokenizer.from_pretrained(MODEL_ID)
MMLU_TOK.pad_token = MMLU_TOK.eos_token
MMLU_TOK.padding_side = "left"

print("Loading MMLU-Pro dataset...")
mmlu_full = load_dataset("TIGER-Lab/MMLU-Pro", split="test", trust_remote_code=True)
subjects  = list(set(mmlu_full["category"]))
per_subj  = max(1, MMLU_N // len(subjects))

random.seed(42)
sampled = []
for subj in subjects:
    subj_qs = [q for q in mmlu_full if q["category"] == subj]
    sampled += random.sample(subj_qs, min(per_subj, len(subj_qs)))
random.shuffle(sampled)
sampled = sampled[:MMLU_N]

sampled_ids  = {q.get("question_id", i) for i, q in enumerate(sampled)}
fewshot_pool = [q for i, q in enumerate(mmlu_full)
                if q.get("question_id", i) not in sampled_ids][:500]

print(f"  Test questions : {len(sampled):,} across {len(subjects)} subjects")
print(f"  Few-shot pool  : {len(fewshot_pool):,}")
print(f"  MAX_NEW_TOK    : {MAX_NEW_TOK}  (single-letter answers)")


def make_fewshot_prefix(pool, category, n=5):
    same = [q for q in pool if q["category"] == category]
    exs  = random.sample(same if len(same) >= n else pool, n)
    prefix = ""
    for ex in exs:
        opts = "\n".join(f"({chr(65+i)}) {o}"
                         for i, o in enumerate(ex["options"]))
        gt = get_answer_letter(ex)
        prefix += f"Q: {ex['question']}\n{opts}\nA: {gt}\n\n"
    return prefix


def format_question(q):
    opts = "\n".join(f"({chr(65+i)}) {o}" for i, o in enumerate(q["options"]))
    return f"Q: {q['question']}\n{opts}\nA:"


def get_answer_letter(q):
    if "answer" in q and isinstance(q["answer"], str):
        return q["answer"].upper().strip()
    if "answer_index" in q and isinstance(q["answer_index"], int):
        return chr(65 + q["answer_index"])
    if "answer_index" in q:
        try:
            return chr(65 + int(q["answer_index"]))
        except (ValueError, TypeError):
            pass
    return "A"


def extract_prediction(generated_text):
    for ch in generated_text.strip().upper():
        if ch.isalpha() and 0 <= ord(ch) - 65 < 10:
            return ch
    return "?"


def eval_mmlu_one_experiment(exp_id, adapter_path):
    """Load one adapter on quantized base, run MMLU-Pro, unload."""
    print(f"\n{'='*55}\n  MMLU: {exp_id}")

    # ── CHANGED: Load 4-bit quantized (FP16 = 16 GB, won't fit on T4) ────────
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="sdpa",
    )

    # ── CHANGED: Load adapter without merging (can't merge into quantized) ────
    if adapter_path and os.path.isdir(adapter_path):
        try:
            model = PeftModel.from_pretrained(base, adapter_path)
            print("  LoRA adapter loaded ✅")
        except Exception as e:
            print(f"  ⚠️  Adapter load failed ({e}) — evaluating base model")
            model = base
    else:
        model = base
        print(f"  ⚠️  No adapter at {adapter_path} — evaluating base model")

    model.eval()
    correct = 0
    total   = 0
    per_cat = {}

    for q in tqdm(sampled, desc=exp_id[:25], leave=True):
        prefix    = make_fewshot_prefix(fewshot_pool, q["category"])
        prompt    = prefix + format_question(q)
        messages  = [{"role": "user", "content": prompt}]
        chat_text = MMLU_TOK.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)

        inp = MMLU_TOK(
            chat_text, return_tensors="pt",
            truncation=True, max_length=2048,
        ).to(DEVICE)

        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=MAX_NEW_TOK,
                do_sample=False,
                pad_token_id=MMLU_TOK.eos_token_id,
                temperature=1.0,
            )

        new_tokens = out[0][inp["input_ids"].shape[1]:]
        generated  = MMLU_TOK.decode(new_tokens, skip_special_tokens=True)
        pred       = extract_prediction(generated)
        gt         = get_answer_letter(q)
        ok         = int(pred == gt)
        correct   += ok
        total     += 1

        cat = q["category"]
        if cat not in per_cat:
            per_cat[cat] = [0, 0]
        per_cat[cat][0] += ok
        per_cat[cat][1] += 1

    acc = correct / max(total, 1)
    print(f"  Accuracy: {acc:.4f}  ({correct}/{total})")

    del model, base
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "exp_id"       : exp_id,
        "adapter_path" : str(adapter_path) if adapter_path else "base_model",
        "mmlu_accuracy": round(acc, 4),
        "correct"      : correct,
        "total"        : total,
        "per_category" : {
            k: {"correct": v[0], "total": v[1],
                "acc": round(v[0] / max(v[1], 1), 4)}
            for k, v in per_cat.items()
        },
    }


MMLU_RESULTS = {}
for exp_id in EXP_IDS:
    adapter_path = os.path.join(CHECKPOINT_DIR, exp_id, "final_adapter")
    MMLU_RESULTS[exp_id] = eval_mmlu_one_experiment(
        exp_id,
        adapter_path if os.path.isdir(adapter_path) else None,
    )

mmlu_out = os.path.join(WRITE_DIR, "mmlu_results.json")
with open(mmlu_out, "w") as f:
    json.dump(MMLU_RESULTS, f, indent=2)
print(f"\nMMLU-Pro results saved → {mmlu_out}")  # ← FIXED typo (was MMIU)

print("\n" + "="*65)
print("MMLU-Pro Summary (5-shot, 500 questions)")
print("="*65)
for eid, r in MMLU_RESULTS.items():
    print(f"  {eid:<35} {r['mmlu_accuracy']:.4f}  ({r['correct']}/{r['total']})")
print("="*65)
print("\n✅ Session 3 complete — save output as Kaggle Dataset: 'sft-mmlu'")


In [ ]:
# CELL 11 — Results analysis: 3 tables + 6 figures  (no GPU needed)
# FIX H: Uses 'MMLU_RESULTS' — original typo 'MMILU_RESULTS' would cause crash here.
# FIX M: SHORT/COLORS already defined in Cell 3 — no NameError.
import json, os, math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Load results — handles both in-session and reloaded from file
if "ALL_RESULTS" not in dir() or not ALL_RESULTS:
    p = os.path.join(WRITE_DIR, "all_training_results.json")
    ALL_RESULTS = json.load(open(p)) if os.path.exists(p) else {}
    if ALL_RESULTS:
        print(f"Loaded training results from file ({len(ALL_RESULTS)} experiments)")

if "MMLU_RESULTS" not in dir() or not MMLU_RESULTS:
    p = os.path.join(WRITE_DIR, "mmlu_results.json")
    MMLU_RESULTS = json.load(open(p)) if os.path.exists(p) else {}
    if MMLU_RESULTS:
        print(f"Loaded MMLU results from file ({len(MMLU_RESULTS)} experiments)")


def get(eid, key, default="—"):
    return ALL_RESULTS.get(eid, {}).get(key, default)

def getm(eid):
    v = MMLU_RESULTS.get(eid, {}).get("mmlu_accuracy", "—")
    return f"{v:.4f}" if isinstance(v, float) else str(v)

def tf(v, d=0.0):
    try:    return float(v)
    except: return d


# ── TABLE 1: Full results ──────────────────────────────────────────────────────
print("TABLE 1 — All 6 experiments")
print("="*115)
print(f"{'Exp':<6} {'Description':<38} {'Val PPL':>8} {'MMLU':>8}"
      f" {'Tok/s':>7} {'VRAM':>7} {'Min':>8} {'Optimizer':>16} {'Attn':>16}")
print("="*115)
for eid in EXP_IDS:
    print(f"{SHORT[eid]:<6} {str(get(eid,'desc',''))[:38]:<38}"
          f" {str(get(eid,'final_val_ppl')):>8} {getm(eid):>8}"
          f" {str(get(eid,'avg_tps')):>7} {str(get(eid,'peak_vram_gb')):>7}"
          f" {str(get(eid,'total_time_min')):>8}"
          f" {str(get(eid,'optimizer')):>16} {str(get(eid,'attention')):>16}")
print("="*115)

# ── TABLE 2: 2×2 factorial ────────────────────────────────────────────────────
e1, e2, e3, e4 = [tf(get(e, "final_val_ppl")) for e in EXP_IDS[:4]]
print("\nTABLE 2 — 2×2 Factorial (Val PPL, lower = better)")
print("="*55)
print(f"{'':24} {'Control (Alpaca)':>15} {'Golden (curated)':>15}")
print(f"{'No curriculum':24} {e1:>15.4f} {e2:>15.4f}")
print(f"{'LFR curriculum':24} {e3:>15.4f} {e4:>15.4f}")
if all(v > 0 for v in [e1, e2, e3, e4]):
    print(f"\n  Data quality effect (E2 vs E1) : {(e1-e2)/e1*100:+.2f}%")
    print(f"  Curriculum effect   (E3 vs E1) : {(e1-e3)/e1*100:+.2f}%")
    print(f"  Full framework      (E4 vs E1) : {(e1-e4)/e1*100:+.2f}%")
    intr  = ((e1-e4)/e1 - (e1-e2)/e1 - (e1-e3)/e1) * 100
    label = "synergy" if intr > 0.5 else "additive" if abs(intr) < 0.5 else "overlap"
    print(f"  Interaction term               : {intr:+.2f}%  ({label})")

# ── TABLE 3: Efficiency ───────────────────────────────────────────────────────
e4r = ALL_RESULTS.get("E4_Golden_LFR", {})
e5r = ALL_RESULTS.get("E5_Golden_LFR_GaLore", {})
e6r = ALL_RESULTS.get("E6_Golden_LFR_xFormers", {})
if e4r:
    print("\nTABLE 3 — Efficiency (E4 vs E5 GaLore vs E6 BetterTransformer)")
    print("="*65)
    print(f"  {'Metric':<22} {'E4 baseline':>14} {'E5 GaLore':>14} {'E6 BetterTrans':>14}")
    print("="*65)
    for lbl, key, use_mmlu in [
        ("Val PPL",       "final_val_ppl",  False),
        ("MMLU-Pro",      None,             True),
        ("Tok/s",         "avg_tps",        False),
        ("Peak VRAM GB",  "peak_vram_gb",   False),
        ("Time min",      "total_time_min", False),
    ]:
        if use_mmlu:
            v4, v5, v6 = getm("E4_Golden_LFR"), getm("E5_Golden_LFR_GaLore"), getm("E6_Golden_LFR_xFormers")
        else:
            v4, v5, v6 = e4r.get(key,"—"), e5r.get(key,"—"), e6r.get(key,"—")
        print(f"  {lbl:<22} {str(v4):>14} {str(v5):>14} {str(v6):>14}")
    print("="*65)

# ── FIGURES ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
cols = [COLORS[e] for e in EXP_IDS]

# Fig 1: PPL bar chart
ax1   = fig.add_subplot(gs[0, 0:2])
ppls  = [tf(get(e, "final_val_ppl")) for e in EXP_IDS]
bars1 = ax1.bar([SHORT[e] for e in EXP_IDS], ppls,
                color=cols, edgecolor="white", linewidth=0.8)
ax1.set_ylabel("Response PPL (lower = better)")
ax1.set_title("Fig 1: Final validation PPL — all 6 experiments")
if any(ppls):
    ax1.set_ylim(0, max(p for p in ppls if p > 0) * 1.18)
for bar, v in zip(bars1, ppls):
    if v:
        ax1.text(bar.get_x() + bar.get_width()/2, v + 0.05,
                 f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.axvline(x=3.5, color="gray", linestyle="--", alpha=0.4)
ax1.text(1.5, (max(ppls)*1.12 if any(ppls) else 8),
         "2×2 factorial", ha="center", fontsize=8, color="gray")
ax1.text(4.5, (max(ppls)*1.12 if any(ppls) else 8),
         "efficiency", ha="center", fontsize=8, color="gray")
ax1.grid(axis="y", alpha=0.3, linestyle="--")
ax1.spines[["top","right"]].set_visible(False)

# Fig 2: PPL convergence curves (E1–E4)
ax2 = fig.add_subplot(gs[0, 2])
for eid in EXP_IDS[:4]:
    hist = get(eid, "ppl_history", [])
    if hist:
        ax2.plot(range(1, len(hist)+1), hist,
                 color=COLORS[eid], marker="o", markersize=4,
                 linewidth=2, label=SHORT[eid])
ax2.set_xlabel("Eval checkpoint")
ax2.set_ylabel("Response Val PPL")
ax2.set_title("Fig 2: PPL convergence (E1–E4)")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, linestyle="--")
ax2.spines[["top","right"]].set_visible(False)

# Fig 3: 2×2 heatmap
ax3 = fig.add_subplot(gs[1, 0])
if all(v > 0 for v in [e1, e2, e3, e4]):
    mat = np.array([[e1, e2], [e3, e4]])
    im  = ax3.imshow(mat, cmap="RdYlGn_r", aspect="auto",
                     vmin=min(e1,e2,e3,e4)*0.95, vmax=max(e1,e2,e3,e4)*1.05)
    ax3.set_xticks([0,1]); ax3.set_xticklabels(["Control\n(Alpaca)", "Golden\n(curated)"])
    ax3.set_yticks([0,1]); ax3.set_yticklabels(["No LFR", "LFR"])
    ax3.set_title("Fig 3: 2×2 heatmap\n(greener = lower PPL = better)")
    for i in range(2):
        for j in range(2):
            ax3.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center",
                     color="white", fontweight="bold", fontsize=13)
    plt.colorbar(im, ax=ax3, shrink=0.8, label="Val PPL")
    ax3.set_xlabel(
        f"Data: {(e1-e2)/e1*100:+.1f}%   "
        f"Curr: {(e1-e3)/e1*100:+.1f}%   "
        f"Full: {(e1-e4)/e1*100:+.1f}%",
        fontsize=8, color="gray")

# Fig 4: MMLU-Pro bar chart
ax4 = fig.add_subplot(gs[1, 1])
mmlus  = [tf(MMLU_RESULTS.get(e, {}).get("mmlu_accuracy", 0)) for e in EXP_IDS]
bars4  = ax4.bar([SHORT[e] for e in EXP_IDS], mmlus,
                 color=cols, edgecolor="white", linewidth=0.8)
ax4.set_ylabel("MMLU-Pro accuracy (5-shot)")
ax4.set_title("Fig 4: MMLU-Pro accuracy")
if any(mmlus):
    ax4.set_ylim(0, max(m for m in mmlus if m > 0) * 1.15)
for i, v in enumerate(mmlus):
    if v:
        ax4.text(i, v + 0.002, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
ax4.grid(axis="y", alpha=0.3, linestyle="--")
ax4.spines[["top","right"]].set_visible(False)

# Fig 5: VRAM vs PPL scatter (E4–E6 efficiency)
ax5 = fig.add_subplot(gs[1, 2])
for eid in ["E4_Golden_LFR", "E5_Golden_LFR_GaLore", "E6_Golden_LFR_xFormers"]:
    v = tf(get(eid, "peak_vram_gb"))
    p = tf(get(eid, "final_val_ppl"))
    if v and p:
        ax5.scatter(v, p, color=COLORS[eid], s=150, zorder=5)
        ax5.annotate(SHORT[eid], (v, p), textcoords="offset points",
                     xytext=(6, 4), fontsize=10, fontweight="bold")
ax5.set_xlabel("Peak VRAM (GB)")
ax5.set_ylabel("Val PPL")
ax5.set_title("Fig 5: VRAM vs PPL\n(E4–E6 efficiency tradeoff)")
ax5.grid(alpha=0.3, linestyle="--")
ax5.spines[["top","right"]].set_visible(False)

# Fig 6: Throughput bar chart (all 6)
ax6   = fig.add_subplot(gs[2, 0:2])
tpss  = [tf(get(e, "avg_tps")) for e in EXP_IDS]
bars6 = ax6.bar([SHORT[e] for e in EXP_IDS], tpss,
                color=cols, edgecolor="white", linewidth=0.8)
ax6.set_ylabel("Avg tokens/sec (higher = faster)")
ax6.set_title("Fig 6: Training throughput — all 6 experiments")
if any(tpss):
    ax6.set_ylim(0, max(t for t in tpss if t > 0) * 1.2)
for bar, v in zip(bars6, tpss):
    if v:
        ax6.text(bar.get_x() + bar.get_width()/2, v + 0.5,
                 f"{v:.0f}", ha="center", va="bottom", fontsize=9)
ax6.grid(axis="y", alpha=0.3, linestyle="--")
ax6.spines[["top","right"]].set_visible(False)

fig.suptitle("SFT Pipeline — LLaMA 3.1 8B on T4 — Full Ablation Results",
             fontsize=14, fontweight="bold", y=0.98)
plt.savefig(os.path.join(WRITE_DIR, "all_results_figures.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"Figures saved → {os.path.join(WRITE_DIR, 'all_results_figures.png')}")

# Save final combined JSON
final_out = os.path.join(WRITE_DIR, "all_results_final.json")
with open(final_out, "w") as f:
    json.dump({"training": ALL_RESULTS, "mmlu": MMLU_RESULTS}, f, indent=2)
print(f"Combined results saved → {final_out}")
print("\n✅ All done — save output as Kaggle Dataset: 'sft-all-results'")
